In [1]:
from typing import Final
from pathlib import Path
from os import getenv
from shutil import copyfile
from dotenv import find_dotenv, load_dotenv
import rasterio as r
from rasterio.merge import merge as merge_raster

PROJECT_DIR: Final[Path] = Path(find_dotenv(".env", 1, 1)).absolute().parent
load_dotenv(PROJECT_DIR.joinpath(".env"))

EDINA_DIR: Final[Path] = Path(getenv("EDINA_DOWNLOAD_DIR"))

In [2]:
tiff_dirs: list[list[Path]] = [
    [*el.glob("*.tif")] for em in EDINA_DIR.iterdir() if em.is_dir()
    for el in em.iterdir() if el.is_dir() and ("edit" not in el.name)
]
# for tiffs in tiff_dirs:
#     if len(tiffs) > 1:
#         # Instance raster readers for the multiple tiff files
#         srcs = [r.open(el, "r") for el in tiffs]
#         # Extract meta data to use as a template
#         meta = srcs[0].meta.copy()
#         # Combine tiff files and transform into a single instance
#         tiff, transform = merge_raster(srcs)
#         for src in srcs:
#             src.close() # close readers
#         # update meta data dictionary with merged file meta
#         meta.update({
#             "height": tiff.shape[1],
#             "width": tiff.shape[2],
#             "transform": transform
#         })
#         # write out merged file
#         with r.open(
#             PROJECT_DIR.joinpath(f"data/tiffs/{tiffs[0].parent.parent.name}"),
#             "w",
#             **meta
#         ) as src:
#             src.write(tiff)
    
#     copyfile(tiffs[0], PROJECT_DIR.joinpath(f"data/tiffs/{tiffs[0].name}"))
#     meta = tiffs[0].parent.joinpath(f"{tiffs[0].stem}.tfw")
#     copyfile(meta, PROJECT_DIR.joinpath(f"data/tiffs/{meta.name}"))

In [3]:
check = True
for tiffs in tiff_dirs:
    for tiff in tiffs:
        with r.open(tiff, "r") as src:
            check &= (src.read().shape == (1, 6000, 6000))

print(f"all files are dimension 6000x6000: {check}")

all files are dimension 6000x6000: True


In [4]:
from pandas import concat
from shapely import Point
from edina import parse_edina_tab_file

tiffs = next(el for el in tiff_dirs if len(el) > 1)
metas = [
    parse_edina_tab_file(tiff.parent.joinpath(f"{tiff.stem}.tab"))
    for tiff in tiffs
]
top_left_boundary = [
    gdf.loc[
        (gdf.geometry.x == gdf.geometry.x.min())
        & (gdf.geometry.y == gdf.geometry.y.max()),
        "geometry"
    ].iloc[0]
    for gdf in metas
]
top_left_global_idx = top_left_boundary.index(Point(
    min(p.x for p in top_left_boundary), max(p.y for p in top_left_boundary)
))
verticals_idx = [
    i
    for i, el
    in sorted(enumerate(top_left_boundary), key = lambda x: x[1].y, reverse=1)
    if el.x == top_left_boundary[top_left_global_idx].x
]
verticals_idx

[0, 2]